In [16]:
import time
import sys
import pandas as pd
from collections import defaultdict

In [17]:
# Sample dataset
documents = [
    "The quick brown fox jumped over the lazy dog",
    "The quick brown fox is fast",
    "A lazy dog is not so quick"
]

search_term = "quick"
BLOCK_SIZE = 4  # words per block

In [18]:
from collections import defaultdict

def build_inverted_index(docs):
    inverted_index = defaultdict(list)
    for doc_id, text in enumerate(docs):
        words = text.lower().split()
        for pos, word in enumerate(words):
            inverted_index[word].append((doc_id, pos))
    return dict(inverted_index)

# Build inverted index
inverted_index = build_inverted_index(documents)

print("Inverted Index:")
for term, postings in inverted_index.items():
    print(f"{term}: {postings}")

Inverted Index:
the: [(0, 0), (0, 6), (1, 0)]
quick: [(0, 1), (1, 1), (2, 6)]
brown: [(0, 2), (1, 2)]
fox: [(0, 3), (1, 3)]
jumped: [(0, 4)]
over: [(0, 5)]
lazy: [(0, 7), (2, 1)]
dog: [(0, 8), (2, 2)]
is: [(1, 4), (2, 3)]
fast: [(1, 5)]
a: [(2, 0)]
not: [(2, 4)]
so: [(2, 5)]


In [19]:
def search_inverted_index(term, inverted_index):
    term = term.lower()
    return inverted_index.get(term, [])

# Search for "quick"
results_inverted = search_inverted_index(search_term, inverted_index)
print(f"Search results for '{search_term}' in Inverted Index: {results_inverted}")

Search results for 'quick' in Inverted Index: [(0, 1), (1, 1), (2, 6)]


In [20]:
from collections import defaultdict

def build_block_addressing(docs, block_size):
    block_index = defaultdict(set)
    blocks = {}  # store block_id → (doc_id, words)
    block_id = 0

    for doc_id, text in enumerate(docs):
        words = text.lower().split()
        for i in range(0, len(words), block_size):
            block_words = words[i:i+block_size]
            blocks[block_id] = (doc_id, block_words)  # store block contents
            for word in block_words:
                block_index[word].add((doc_id, block_id))
            block_id += 1

    return {word: list(blocks_set) for word, blocks_set in block_index.items()}, blocks


# Build block addressing index + blocks
block_index, blocks = build_block_addressing(documents, BLOCK_SIZE)

print("=== Block Addressing Index ===")
for term, block_list in block_index.items():
    print(f"{term}: {block_list}")

print("\n=== Blocks ===")
for block_id, (doc_id, words) in blocks.items():
    print(f"Block {block_id} (Doc {doc_id}): {words}")

=== Block Addressing Index ===
the: [(0, 1), (1, 3), (0, 0)]
quick: [(1, 3), (2, 6), (0, 0)]
brown: [(1, 3), (0, 0)]
fox: [(1, 3), (0, 0)]
jumped: [(0, 1)]
over: [(0, 1)]
lazy: [(0, 1), (2, 5)]
dog: [(0, 2), (2, 5)]
is: [(2, 5), (1, 4)]
fast: [(1, 4)]
a: [(2, 5)]
not: [(2, 6)]
so: [(2, 6)]

=== Blocks ===
Block 0 (Doc 0): ['the', 'quick', 'brown', 'fox']
Block 1 (Doc 0): ['jumped', 'over', 'the', 'lazy']
Block 2 (Doc 0): ['dog']
Block 3 (Doc 1): ['the', 'quick', 'brown', 'fox']
Block 4 (Doc 1): ['is', 'fast']
Block 5 (Doc 2): ['a', 'lazy', 'dog', 'is']
Block 6 (Doc 2): ['not', 'so', 'quick']


In [21]:
def search_block_addressing(term, block_index):
    term = term.lower()
    return block_index.get(term, [])

# Search for "quick"
results_block = search_block_addressing(search_term, block_index)
print(f"Search results for '{search_term}' in Block Addressing: {results_block}")

Search results for 'quick' in Block Addressing: [(1, 3), (2, 6), (0, 0)]


In [22]:
def build_suffix_array(text):
    suffixes = [(text[i:], i) for i in range(len(text))]
    suffixes.sort()
    return [pos for (suff, pos) in suffixes]

data = "banana$"
suffix_array = build_suffix_array(data)

print("Suffix Array for 'banana$':")
print(suffix_array)

Suffix Array for 'banana$':
[6, 5, 3, 1, 0, 4, 2]


In [23]:
print("=== Final Results ===")
print(f"Inverted Index Search for '{search_term}': {results_inverted}")
print(f"Block Addressing Search for '{search_term}': {results_block}")
print(f"Suffix Array for 'banana$': {suffix_array}")

=== Final Results ===
Inverted Index Search for 'quick': [(0, 1), (1, 1), (2, 6)]
Block Addressing Search for 'quick': [(1, 3), (2, 6), (0, 0)]
Suffix Array for 'banana$': [6, 5, 3, 1, 0, 4, 2]


In [24]:
def get_size(obj):
    """Recursively finds size of objects in bytes"""
    seen_ids = set()
    size = 0
    objects = [obj]
    while objects:
        current = objects.pop()
        if id(current) not in seen_ids:
            seen_ids.add(id(current))
            size += sys.getsizeof(current)
            if isinstance(current, dict):
                objects.extend(current.keys())
                objects.extend(current.values())
            elif isinstance(current, (list, set, tuple)):
                objects.extend(current)
    return size

In [25]:
# Measure time for index creation
start_time = time.time()
inverted_index = build_inverted_index(documents)
build_time_inverted = time.time() - start_time

# Measure memory usage
memory_inverted = get_size(inverted_index)

# Search times
def phrase_search_inverted(phrase, inverted_index, docs):
    words = phrase.lower().split()
    postings = [inverted_index.get(w, []) for w in words]
    results = []
    for doc_id, _ in postings[0]:
        for pos in [p for d, p in postings[0] if d == doc_id]:
            if all((doc_id, pos+i) in postings[i] for i in range(len(words))):
                results.append((doc_id, pos))
    return results

queries = {
    "single": "quick",
    "multi": ["quick", "dog"],
    "phrase": "quick brown fox"
}

# Execution times
times_inverted = {}
for qtype, q in queries.items():
    start_time = time.time()
    if qtype == "single":
        search_inverted_index(q, inverted_index)
    elif qtype == "multi":
        [search_inverted_index(term, inverted_index) for term in q]
    else:  # phrase
        phrase_search_inverted(q, inverted_index, documents)
    times_inverted[qtype] = time.time() - start_time

In [26]:
# Measure time for index creation
start_time = time.time()
block_index, blocks = build_block_addressing(documents, BLOCK_SIZE)
build_time_block = time.time() - start_time

# Measure memory usage
memory_block = get_size(block_index)

# Execution times
times_block = {}
for qtype, q in queries.items():
    start_time = time.time()
    if qtype == "single":
        search_block_addressing(q, block_index)
    elif qtype == "multi":
        [search_block_addressing(term, block_index) for term in q]
    else:  # phrase (simulate by intersecting blocks containing all words)
        words = q.split()
        results = set(search_block_addressing(words[0], block_index))
        for w in words[1:]:
            results &= set(search_block_addressing(w, block_index))
    times_block[qtype] = time.time() - start_time

In [27]:
data = "banana$"

# Measure time for index creation
start_time = time.time()
suffix_array = build_suffix_array(data)
build_time_suffix = time.time() - start_time

# Measure memory usage
memory_suffix = get_size(suffix_array)

# Execution times for search
def search_suffix_array(pattern, text, suffix_array):
    results = []
    for pos in suffix_array:
        if text.startswith(pattern, pos):
            results.append(pos)
    return results

queries_suffix = {
    "single": "ana",
    "multi": ["ana", "na"],  # simulate by running multiple searches
    "phrase": "banana"
}

times_suffix = {}
for qtype, q in queries_suffix.items():
    start_time = time.time()
    if qtype == "single":
        search_suffix_array(q, data, suffix_array)
    elif qtype == "multi":
        [search_suffix_array(term, data, suffix_array) for term in q]
    else:  # phrase
        search_suffix_array(q, data, suffix_array)
    times_suffix[qtype] = time.time() - start_time

In [28]:
df = pd.DataFrame({
    "Index Type": ["Inverted Files", "Block Addressing", "Suffix Arrays"],
    "Build Time (s)": [build_time_inverted, build_time_block, build_time_suffix],
    "Memory (bytes)": [memory_inverted, memory_block, memory_suffix],
    "Single-word Query (s)": [times_inverted["single"], times_block["single"], times_suffix["single"]],
    "Multi-word Query (s)": [times_inverted["multi"], times_block["multi"], times_suffix["multi"]],
    "Phrase Query (s)": [times_inverted["phrase"], times_block["phrase"], times_suffix["phrase"]]
})

df

,Index Type,Build Time (s),Memory (bytes),Single-word Query (s),Multi-word Query (s),Phrase Query (s)
0,Inverted Files,0.000142,3670,0.000005,0.000003,0.000034
1,Block Addressing,0.000172,3438,0.000004,0.000003,0.000011
2,Suffix Arrays,0.000113,316,0.000006,0.000006,0.000003
